# Auditoría Drive — Carpeta de Rocío

Escanea las carpetas de Rocío en Drive para encontrar sus resultados de entrenamiento
y poder incluirlos en la tabla comparativa.

No borra ni modifica nada — solo lista y lee metricas.json si existen.

In [ ]:
from google.colab import drive
import os
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
# ── Buscar carpetas de Rocío en todas las rutas posibles ──────
from pathlib import Path
import json

DRIVE = '/content/drive/MyDrive'

def tamanyo(n):
    for u in ['B','KB','MB','GB']:
        if n < 1024: return f'{n:.0f}{u}'
        n /= 1024
    return f'{n:.1f}TB'

# Nombres de carpeta que puede usar Rocío
NOMBRES_POSIBLES = [
    'Rocio', 'Rocío', 'rocio', 'ROCIO',
    'E3_Rocio', 'E3_Rocío', 'E3-Rocio',
    'Rocio_E3', 'modelo_rocio', 'resultados_rocio',
]

# Directorios raíz donde buscar
RAICES = [
    f'{DRIVE}/Datos_E2_E3/E3',
    f'{DRIVE}/Datos_E2_E3',
    f'{DRIVE}',
]

encontradas = []

print('Buscando carpetas de Rocío...')
for raiz in RAICES:
    p = Path(raiz)
    if not p.exists(): continue
    try:
        for hijo in p.iterdir():
            if hijo.is_dir():
                nombre_lower = hijo.name.lower()
                if any(n.lower() in nombre_lower for n in NOMBRES_POSIBLES):
                    encontradas.append(hijo)
                    print(f'  [ENCONTRADA] {hijo}')
    except PermissionError:
        pass

if not encontradas:
    print('  No se encontró ninguna carpeta con nombre de Rocío.')
    print('  Comprueba si usa otro nombre — ejecuta la celda de abajo para listar todo E3/')

In [ ]:
# ── Listar TODO lo que hay en Datos_E2_E3/E3/ ────────────────
# Ejecuta esto si la celda anterior no encontró nada

from pathlib import Path

DRIVE = '/content/drive/MyDrive'
e3 = Path(f'{DRIVE}/Datos_E2_E3/E3')

def tamanyo(n):
    for u in ['B','KB','MB','GB']:
        if n < 1024: return f'{n:.0f}{u}'
        n /= 1024
    return f'{n:.1f}TB'

if e3.exists():
    print(f'Contenido de {e3}:')
    for hijo in sorted(e3.iterdir()):
        if hijo.is_dir():
            try:
                n = len(list(hijo.iterdir()))
                # Intentar contar subcarpetas
                subcarpetas = [x for x in hijo.iterdir() if x.is_dir()]
                print(f'  📁 {hijo.name}/  ({n} elementos, {len(subcarpetas)} subcarpetas)')
                for sub in sorted(subcarpetas)[:5]:  # max 5 para no flood
                    try:
                        ns = len(list(sub.iterdir()))
                        print(f'       📁 {sub.name}/  ({ns} elementos)')
                    except: pass
            except: print(f'  📁 {hijo.name}/  [error]')
        else:
            print(f'  📄 {hijo.name}')
else:
    print(f'No existe: {e3}')
    print('Intentando listar MyDrive raíz...')
    for hijo in sorted(Path(DRIVE).iterdir()):
        print(f'  {"📁" if hijo.is_dir() else "📄"} {hijo.name}')

In [ ]:
# ── Escanear en detalle las carpetas de Rocío encontradas ─────
# Ajusta RUTAS_ROCIO si sabes dónde está su carpeta

from pathlib import Path
import json
from datetime import datetime

DRIVE = '/content/drive/MyDrive'

# ── AJUSTA ESTO si sabes la ruta exacta de Rocío ──────────────
RUTAS_ROCIO = [
    # Descomenta/añade la ruta correcta:
    # f'{DRIVE}/Datos_E2_E3/E3/Rocio',
    # f'{DRIVE}/Datos_E2_E3/E3/Rocío',
] + [str(p) for p in encontradas]  # usa las encontradas automáticamente

def tamanyo(n):
    for u in ['B','KB','MB','GB']:
        if n < 1024: return f'{n:.0f}{u}'
        n /= 1024
    return f'{n:.1f}TB'

def fecha(p):
    try: return datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d')
    except: return '?'

resultados_rocio = []  # acumula metricas.json encontrados

for ruta_str in RUTAS_ROCIO:
    ruta = Path(ruta_str)
    if not ruta.exists():
        print(f'[NO EXISTE] {ruta}')
        continue

    print(f'\n{"="*60}')
    print(f'Carpeta: {ruta}')
    print('='*60)

    for hijo in sorted(ruta.rglob('*')):
        rel = hijo.relative_to(ruta)
        prof = len(rel.parts) - 1
        ind = '  ' * prof

        if hijo.is_dir():
            try: n = len(list(hijo.iterdir()))
            except: n = '?'
            print(f'{ind}📁 {hijo.name}/  ({n} elem)')
        elif hijo.suffix == '.pt':
            try: sz = tamanyo(hijo.stat().st_size)
            except: sz = '?'
            print(f'{ind}📦 {hijo.name}  {sz}  {fecha(hijo)}')
        elif hijo.name == 'metricas.json':
            try:
                m = json.loads(hijo.read_text())
                print(f'{ind}📊 metricas.json → version={m.get("version")}  '
                      f'CD={m.get("cd_l1")}  F={m.get("f_score")}  '
                      f'ep={m.get("best_epoch")}  test={m.get("n_test")}')
                resultados_rocio.append(m)
            except Exception as e:
                print(f'{ind}📊 metricas.json  [error: {e}]')
        elif hijo.suffix in ('.csv', '.txt', '.png', '.npy', '.ply', '.ipynb'):
            try: sz = tamanyo(hijo.stat().st_size)
            except: sz = '?'
            print(f'{ind}📄 {hijo.name}  {sz}')

print(f'\nTotal metricas.json encontrados: {len(resultados_rocio)}')
if not RUTAS_ROCIO:
    print('⚠️  No se encontraron rutas de Rocío. Ejecuta la Celda 2 primero o edita RUTAS_ROCIO manualmente.')

In [ ]:
# ── Resumen para copiar a Claude ─────────────────────────────
# Escanea con profundidad controlada (sin rglob) — rápido en Drive

from datetime import datetime
import json
from pathlib import Path

DRIVE = '/content/drive/MyDrive'
BASE  = f'{DRIVE}/Datos_E2_E3'

def tamanyo(n):
    for u in ['B','KB','MB','GB']:
        if n < 1024: return f'{n:.0f}{u}'
        n /= 1024
    return f'{n:.1f}TB'

def listar_nivel(carpeta, profundidad=0, max_prof=3):
    """Lista un directorio sin recursión descontrolada."""
    lineas = []
    ind = '  ' * profundidad
    try:
        hijos = sorted(Path(carpeta).iterdir())
    except Exception as e:
        return [f'{ind}[error: {e}]']

    archivos = [h for h in hijos if h.is_file()]
    subdirs  = [h for h in hijos if h.is_dir()]

    # Agrupar archivos por extensión con tamaño total
    if archivos:
        por_ext = {}
        total_sz = 0
        for f in archivos:
            try: sz = f.stat().st_size
            except: sz = 0
            total_sz += sz
            ext = f.suffix.lower() or '(sin ext)'
            por_ext.setdefault(ext, []).append(f.name)
        ext_resumen = ', '.join(f'{len(v)}x{k}' for k,v in sorted(por_ext.items()))
        lineas.append(f'{ind}  📄 {len(archivos)} archivos [{ext_resumen}] — {tamanyo(total_sz)}')
        # Mostrar archivos individuales si parecen resultados
        for f in archivos:
            ext = f.suffix.lower()
            if ext in ('.json','.csv','.txt') or 'metrica' in f.name.lower() or 'result' in f.name.lower():
                try: sz = tamanyo(f.stat().st_size)
                except: sz = '?'
                # Si es JSON, intentar leer CD/F
                extra = ''
                if ext == '.json':
                    try:
                        d = json.loads(f.read_text())
                        if 'cd_l1' in d or 'f_score' in d:
                            extra = f'  → CD={d.get("cd_l1")}  F={d.get("f_score")}  ep={d.get("best_epoch")}'
                    except: pass
                lineas.append(f'{ind}    🔍 {f.name}  ({sz}){extra}')

    # Subcarpetas
    for d in subdirs:
        try: n = len(list(d.iterdir()))
        except: n = '?'
        lineas.append(f'{ind}  📁 {d.name}/  ({n} elementos)')
        if profundidad < max_prof - 1:
            lineas.extend(listar_nivel(d, profundidad + 1, max_prof))

    return lineas


lineas = []
lineas.append('=== AUDITORÍA DRIVE ROCÍO ===')
lineas.append(f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

# 1. Listar E3/ un nivel para encontrar la carpeta de Rocío
lineas.append('')
lineas.append('--- CONTENIDO DE Datos_E2_E3/E3/ ---')
e3 = Path(f'{BASE}/E3')
carpetas_e3 = []
if e3.exists():
    try:
        for hijo in sorted(e3.iterdir()):
            if hijo.is_dir():
                try: n = len(list(hijo.iterdir()))
                except: n = '?'
                lineas.append(f'  📁 {hijo.name}/  ({n} elementos)')
                carpetas_e3.append(hijo)
            else:
                lineas.append(f'  📄 {hijo.name}')
    except Exception as ex:
        lineas.append(f'  [error: {ex}]')
else:
    lineas.append(f'  [NO EXISTE] {e3}')

# 2. Escanear en detalle todas las carpetas que NO sean "Raquel"
lineas.append('')
lineas.append('--- DETALLE CARPETAS (excluye Raquel) ---')
excluir = {'raquel'}
for carpeta in carpetas_e3:
    if carpeta.name.lower() in excluir:
        continue
    lineas.append(f'')
    lineas.append(f'  📂 {carpeta.name}/')
    lineas.extend(listar_nivel(carpeta, profundidad=1, max_prof=4))

# 3. También listar E3/Raquel brevemente (un nivel)
raquel = e3 / 'Raquel'
if raquel.exists():
    lineas.append('')
    lineas.append('--- E3/Raquel/ (referencia — un nivel) ---')
    try:
        for h in sorted(raquel.iterdir()):
            if h.is_dir():
                try: n = len(list(h.iterdir()))
                except: n = '?'
                lineas.append(f'  📁 {h.name}/  ({n} elementos)')
    except: pass

lineas.append('')
lineas.append('=== FIN AUDITORÍA ===')

txt = '\n'.join(lineas)
print(txt)
print()
print('↑ Copia todo el texto y pégalo en Claude.')